# 🔬 Cryo-EM & SAXS Lab: Visualizing Conformational Heterogeneity
### Simulating Global Structural Observables for Synthetic Ensembles

---

## 🎯 Learning Objectives
In this lab, we explore how protein flexibility and resolution limits manifest in two critical structural biology techniques:
1. **SAXS (Small-Angle X-ray Scattering)**: How global shape and solution dynamics dictate scattering profiles.
2. **Cryo-EM (Cryogenic Electron Microscopy)**: How conformational heterogeneity and resolution blurring affect 3D density maps.

By the end of this tutorial, you will be able to generate synthetic "ground truth" ensembles and simulate the corresponding experimental data that a structural biologist would see in the lab.

In [ ]:
# 🔧 Environment Setup
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q synth-pdb biotite mrcfile matplotlib numpy scipy
else:
    sys.path.append(os.path.abspath('../../'))

from synth_pdb.cryo_em import generate_density_map, save_mrc_file
from synth_pdb.generator import PeptideGenerator
from synth_pdb.saxs import SaxsSimulator, calculate_saxs_profile

print("✅ Environment configured!")

## 1. Generating the "Flexible" Ensemble

We will use a 30-residue sequence and generate an ensemble of "random coil" conformations. This mimics an **Intrinsically Disordered Protein (IDP)**, which doesn't have a single fixed structure but exists as a collection of diverse states.

In [ ]:
seq = "DSHAKRHHGYKRKFHEKHHSHRGYRSNYLY" # Histatin 5-like sequence
gen = PeptideGenerator(seq)

print(f"Generating an ensemble of 50 models for sequence: {seq}")
ensemble = gen.generate_ensemble(n_models=50, conformation="random")

print(f"✅ Generated ensemble with {len(ensemble)} models.")

## 2. SAXS: Probing Global Shape

Small-Angle X-ray Scattering (SAXS) measures the scattering of X-rays by proteins in solution. For an ensemble, the measured signal is the **average** of the scattering from all individual molecules.

### The Debye Formula
We use the Debye formula to compute the intensity $I(q)$:
$$I(q) = \sum_i \sum_j f_i(q) f_j(q) \frac{\sin(q r_{ij})}{q r_{ij}}$$

Let's compare the SAXS profile of a **single static model** vs. the **ensemble average**.

In [ ]:
simulator = SaxsSimulator(q_min=0.0, q_max=0.5, n_points=50)

# 1. Calculate for a single model
q, single_intensity = calculate_saxs_profile(ensemble[0], q_max=0.5, n_points=50)

# 2. Calculate for the whole ensemble (Average)
ensemble_intensity = simulator.simulate(ensemble)

# Plotting
plt.figure(figsize=(8, 5))
plt.semilogy(q, single_intensity, 'r--', label='Single Model')
plt.semilogy(q, ensemble_intensity, 'b-', linewidth=2, label='Ensemble Average')
plt.xlabel('q (Å⁻¹)', fontsize=12)
plt.ylabel('log I(q)', fontsize=12)
plt.title('SAXS Profile: Single Model vs. Ensemble Average', fontsize=14)
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

print("Note how the ensemble average is smoother, reflecting the averaged structural diversity.")

## 3. Cryo-EM: Visualizing Resolution and Heterogeneity

Cryo-EM density maps represent the Coulomb potential of the sample. Unlike the "sticks and balls" of a PDB file, a Cryo-EM map is a 3D grid (voxels) of density values.

### Experiment A: Resolution Limits
We'll generate maps for the **same structure** at different resolutions to see how detail is lost.

In [ ]:
model = ensemble[0]

# Generate 3Å resolution map (High res)
density_3a, _ = generate_density_map(model, resolution=3.0)

# Generate 8Å resolution map (Low res)
density_8a, _ = generate_density_map(model, resolution=8.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.max(density_3a, axis=0), cmap='viridis')
axes[0].set_title('3Å Resolution (Atomic Details)')
axes[1].imshow(np.max(density_8a, axis=0), cmap='viridis')
axes[1].set_title('8Å Resolution (Envelopes)')
plt.show()

### Experiment B: The "Blurred" Ensemble
When we generate a map from an **ensemble**, mobile regions of the protein will naturally appear at lower density and more "blurred" because they occupy different voxels in each model. This is exactly how flexible loops appear in real Cryo-EM experiments!

In [ ]:
# Generate map from the whole ensemble at 4Å resolution
ensemble_density, origin = generate_density_map(ensemble, resolution=4.0)

plt.figure(figsize=(6, 6))
plt.imshow(np.max(ensemble_density, axis=0), cmap='magma')
plt.title('Ensemble-Averaged Cryo-EM Map (4Å)')
plt.colorbar(label='Density Value')
plt.show()

save_mrc_file("ensemble_map.mrc", ensemble_density, origin)
print("✅ Saved ensemble map to ensemble_map.mrc")

## 4. Summary: Integrative Modeling

In this lab, we've seen how `synth-pdb` allows us to move from abstract coordinates to realistic experimental observables:
- **SAXS** captures the global "envelope" and spatial distribution of atoms.
- **Cryo-EM** captures the 3D volume and local resolution effects.

By using both, researchers can "triangulate" the true structure of a protein, even when it is highly flexible or difficult to crystallize.